# Tracking Data Cleanup

Ported from [JohnComonitski/FootballTrackingDataGeneration](https://github.com/JohnComonitski/FootballTrackingDataGeneration/tree/main/data_cleanup/lib) and adapted to the `foostats_ai` tracking CSV schema.

**Schema** (`Match.import_raw_data`): `Frame, Object, Object ID, Team, X1, Y1, X2, Y2, X_Pitch, Y_Pitch, X_MPLSoccer, Y_MPLSoccer`

- `X_Pitch` / `Y_Pitch` — field coordinates in meters (from the view transformer's `position_transformed`).
- `X_MPLSoccer` / `Y_MPLSoccer` — those values normalized to `[0, 1]` by the standard 105 x 68 pitch, ready for mplsoccer.

What the cleanup provides:
- `Ball.clean_path()` — lone-point + nearby filtering, gap interpolation, and (when `filterpy` is installed) Kalman arc removal.
- `Match.merge_players()` / `remove_player()` — fix tracker ID switches and drop spurious tracks.
- `Player.change_team()` / `change_name()` — correct team-classification mistakes.
- `Ball.plot()` / `Player.plot()` — sanity-check trajectories.
- `Match.export()` — write the cleaned CSV back out (to `./output/`).

In [ ]:
import os, sys

# Make `from lib.match import Match` resolve regardless of where the kernel started.
NB_DIR = os.path.dirname(os.path.abspath("cleanup.ipynb"))
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

import matplotlib
import matplotlib.pyplot as plt
from lib.match import Match
from lib.player import Player
print("imports ok")

## (Optional) Generate the tracking CSV from the pipeline

After `main()` runs it exposes `main._debug_all_tracks`. Convert it to the cleanup schema with `export_tracks.tracks_to_csv`. This notebook ships with a synthetic `sample_tracks.csv` so it runs standalone, but in practice you would do:

```python
import main
from export_tracks import tracks_to_csv
main.main()                       # runs the full detection/tracking pipeline
tracks_to_csv(main._debug_all_tracks, "tracks.csv")
```

In [ ]:
CSV_NAME = "sample_tracks.csv"  # swap for your pipeline-exported CSV
FPS = 25  # source video frame rate; the pipeline's tracker runs at 25

match = Match()
match.import_raw_data("./", CSV_NAME, fps=FPS)
print("source:", match.source)
print("ball frames:", match.frames)
print("players:", [(p.id, p.team) for p in match.players])

## Clean the ball path

Removes spurious lone detections, drops far jumps, and interpolates short gaps. If `filterpy` is installed the Kalman arc-removal step also runs; otherwise it is skipped (a warning-free no-op) and the rest still applies.

In [ ]:
ball = match.ball
missing_before = sum(1 for f in ball.frames if f.coordinates is None)
ball.clean_path()
missing_after = sum(1 for f in ball.frames if f.coordinates is None)
print(f"ball missing frames: {missing_before} -> {missing_after}")

ball.plot()  # saves 'Ball Path.png'
plt.show()

## Fix player tracks

Typical corrections after tracking:
- `change_team(...)` when the color classifier put a player on the wrong team.
- `change_name(...)` to label a track.
- `merge_players(a, b)` when one player got split into two IDs (ID switch).
- `remove_player(p)` to drop a phantom/duplicate track.

In [ ]:
p7 = match.player("7")
p7.change_name("Striker")
p7.plot()  # saves 'Striker Path.png'
plt.show()

# Example: correct a misclassified team
# p7.change_team(2.0)

# Example: merge an ID switch (merges p2's frames into p1 where p1 is missing)
# merged = match.merge_players(match.player("7"), match.player("12"))

# Example: drop a phantom track
# match.remove_player(match.player("99"))
print("players now:", [(p.id, p.name, p.team) for p in match.players])

## Export the cleaned data

Writes `./output/<CSV_NAME>` in the same schema (including the `X_MPLSoccer` / `Y_MPLSoccer` columns), recomputed from the cleaned coordinates.

In [ ]:
match.export()
out = os.path.join("output", CSV_NAME)
print("wrote", out)
with open(out) as f:
    for line in list(f)[:4]:
        print(line.rstrip())